# SkyPose — chạy thử nhận diện tư thế trên ảnh

Notebook nạp checkpoint đã train (`checkpoints/resnet18/best.pt`) và chạy suy luận trên ảnh tĩnh:

1. Nạp model qua `PoseEngine` (dùng chung code với server: flip-test, FP16 trên GPU).
2. Suy luận **2 lượt**: lượt 1 trên cả ảnh, lượt 2 cắt lại vùng quanh người vừa tìm được (giống tracker của webcam) để tăng độ chính xác.
3. Vẽ khung xương, bảng độ tin cậy từng khớp, heatmap đầu ra.
4. So với nhãn COCO val2017 (nếu có dữ liệu) và lưu ảnh kết quả.

> Model là **top-down 1 người**: ảnh nên có một người là chính. Ảnh nhiều người chỉ cho ra khung xương của 1 người.

Chạy bằng kernel của `.venv` (cần `ipykernel`: `.venv/bin/pip install ipykernel`).

In [ ]:
import sys
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

# tìm thư mục gốc repo (notebook nằm ở notebooks/)
ROOT = Path.cwd()
while not (ROOT / "training" / "pose_utils.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Không tìm thấy thư mục gốc repo (cần training/pose_utils.py)")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "training"))

from server.pose_engine import PoseEngine  # noqa: E402
from pose_utils import (FLIP_INDEX, INPUT_SIZE, KEYPOINTS, OKS_SIGMAS, SKELETON,  # noqa: E402
                        square_affine, to_tensor_input)

print("ROOT  :", ROOT)
print("torch :", torch.__version__, "| CUDA:", torch.cuda.is_available())

## Cấu hình

- `CKPT`: checkpoint cần thử. Mặc định lấy `runs/*/resnet18/best.pt` mới nhất (bản train đủ 20 epoch trên Kaggle);
  `checkpoints/resnet18/best.pt` ở máy local chỉ là bản train dở (epoch 3).
- `IMAGE_DIR`: thư mục chứa ảnh của bạn (jpg/png). Nếu trống/không tồn tại, notebook lấy vài ảnh COCO val2017 có đúng 1 người.
- `CONF_THR`: ngưỡng độ tin cậy để vẽ một khớp.

In [ ]:
_runs = sorted(ROOT.glob("runs/*/resnet18/best.pt"))
CKPT = _runs[-1] if _runs else ROOT / "checkpoints" / "resnet18" / "best.pt"
IMAGE_DIR = ROOT / "test_images"          # bỏ ảnh của bạn vào đây
OUT_DIR = ROOT / "logs" / "test_images"   # nơi lưu ảnh kết quả
CONF_THR = 0.3
FLIP_TEST = True
DEVICE = None                             # None = tự chọn cuda/cpu; hoặc "cpu"
N_COCO_SAMPLES = 6                        # số ảnh COCO dùng khi không có ảnh riêng

COCO_IMG_DIR = ROOT / "data" / "coco" / "val2017"
COCO_ANN = ROOT / "data" / "coco" / "annotations" / "person_keypoints_val2017.json"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Nạp model

In [ ]:
assert CKPT.exists(), f"Không thấy checkpoint: {CKPT}"
t0 = time.perf_counter()
engine = PoseEngine(CKPT, device=DEVICE, flip=FLIP_TEST)
print(f"Nạp xong trong {time.perf_counter() - t0:.2f}s")
for k, v in engine.info.items():
    if k != "keypoints":
        print(f"  {k:11s}: {v}")
n_params = sum(p.numel() for p in engine.model.parameters())
print(f"  params     : {n_params / 1e6:.2f}M")

## Hàm suy luận và vẽ

`detect()` chạy 2 lượt: lượt 1 trên hình vuông bao cả ảnh; nếu thấy đủ khớp thân trên, lượt 2 cắt hình vuông
quanh các khớp tìm được (nới 1.6 lần như `PoseSession` của server, nhưng tính cả chân) rồi chạy lại.

In [ ]:
TRACK_KPTS = list(range(17))  # ảnh tĩnh: dùng cả chân (server chỉ dùng 0..12 cho webcam)


def full_box(img):
    h, w = img.shape[:2]
    return (w / 2, h / 2, float(max(h, w)))


def box_from_kp(kp, img, conf_thr=CONF_THR):
    # vùng vuông quanh các khớp tin cậy, None nếu không đủ khớp
    h, w = img.shape[:2]
    good = [j for j in TRACK_KPTS if kp[j, 2] > conf_thr]
    if len(good) < 4 or kp[[5, 6], 2].min() <= conf_thr:
        return None
    lo, hi = kp[good, :2].min(0), kp[good, :2].max(0)
    side = float(np.clip((hi - lo).max() * 1.6, 0.35 * max(h, w), 1.6 * max(h, w)))
    return (float(lo[0] + hi[0]) / 2, float(lo[1] + hi[1]) / 2, side)


def detect(img, box=None, refine=True):
    # img: BGR. Trả về kp (17,3) [x, y, conf] theo pixel ảnh gốc, box đã dùng, thời gian (ms)
    t0 = time.perf_counter()
    box = box or full_box(img)
    kp = engine.predict_crop(img, box[:2], box[2])
    if refine:
        nb = box_from_kp(kp, img)
        if nb is not None:
            box = nb
            kp = engine.predict_crop(img, box[:2], box[2])
    if engine.device == "cuda":
        torch.cuda.synchronize()
    return kp, box, (time.perf_counter() - t0) * 1000


LIMB_COLORS = plt.cm.rainbow(np.linspace(0, 1, len(SKELETON)))


def draw_pose(ax, img, kp, box=None, conf_thr=CONF_THR, title=None, gt=None):
    ax.imshow(img[:, :, ::-1])
    if box is not None:
        cx, cy, s = box
        ax.add_patch(plt.Rectangle((cx - s / 2, cy - s / 2), s, s, fill=False, ec="yellow", ls="--", lw=1))
    if gt is not None:  # nhãn COCO (x, y, v)
        for a, b in SKELETON:
            if gt[a, 2] > 0 and gt[b, 2] > 0:
                ax.plot(gt[[a, b], 0], gt[[a, b], 1], color="white", lw=1, ls=":")
        vis = gt[:, 2] > 0
        ax.scatter(gt[vis, 0], gt[vis, 1], s=12, c="white", marker="x", lw=1)
    for (a, b), c in zip(SKELETON, LIMB_COLORS):
        if kp[a, 2] > conf_thr and kp[b, 2] > conf_thr:
            ax.plot(kp[[a, b], 0], kp[[a, b], 1], color=c, lw=2.5)
    ok = kp[:, 2] > conf_thr
    ax.scatter(kp[ok, 0], kp[ok, 1], s=22, c=kp[ok, 2], cmap="viridis", vmin=0, vmax=1,
               edgecolors="black", linewidths=0.6, zorder=3)
    ax.set_xlim(0, img.shape[1]); ax.set_ylim(img.shape[0], 0)
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=9)


def load_bgr(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Không đọc được ảnh: {path}")
    return img

## Chọn ảnh test

Ưu tiên ảnh trong `IMAGE_DIR`; nếu không có thì lấy ngẫu nhiên ảnh COCO val2017 chỉ có **1 người** được gán
≥ 10 khớp (kèm nhãn gốc để so sánh).

In [ ]:
import json
import random

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
images = []  # list of (path, gt_or_None)

if IMAGE_DIR.is_dir():
    images = [(p, None) for p in sorted(IMAGE_DIR.iterdir()) if p.suffix.lower() in EXTS]

coco = None
if COCO_ANN.exists():
    coco = json.loads(COCO_ANN.read_text())
    anns_by_img = {}
    for a in coco["annotations"]:
        if not a.get("iscrowd", 0):
            anns_by_img.setdefault(a["image_id"], []).append(a)
    file_of = {im["id"]: im["file_name"] for im in coco["images"]}

if not images and coco is not None:
    single = [iid for iid, anns in anns_by_img.items()
              if len(anns) == 1 and anns[0]["num_keypoints"] >= 10]
    random.seed(0)
    for iid in random.sample(single, min(N_COCO_SAMPLES, len(single))):
        gt = np.array(anns_by_img[iid][0]["keypoints"], np.float32).reshape(17, 3)
        images.append((COCO_IMG_DIR / file_of[iid], {"kp": gt, "area": anns_by_img[iid][0]["area"]}))
    print(f"IMAGE_DIR trống -> dùng {len(images)} ảnh COCO val2017 (1 người)")

assert images, f"Không có ảnh nào: thêm ảnh vào {IMAGE_DIR} hoặc tải COCO val2017"
for p, _ in images:
    print(" ", p.relative_to(ROOT) if p.is_relative_to(ROOT) else p)

## Chạy nhận diện

In [ ]:
results = []
for path, gt in images:
    img = load_bgr(path)
    kp, box, ms = detect(img)
    results.append({"path": path, "img": img, "kp": kp, "box": box, "ms": ms, "gt": gt})
    print(f"{path.name:28s} {img.shape[1]}x{img.shape[0]}  {ms:6.1f} ms  "
          f"khớp > {CONF_THR}: {(kp[:, 2] > CONF_THR).sum():2d}/17  conf TB: {kp[:, 2].mean():.2f}")

ncol = min(3, len(results))
nrow = int(np.ceil(len(results) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 5 * nrow), squeeze=False)
for ax, r in zip(axes.flat, results):
    draw_pose(ax, r["img"], r["kp"], r["box"], gt=r["gt"]["kp"] if r["gt"] else None,
              title=f"{r['path'].name}  ({r['ms']:.0f} ms)")
for ax in axes.flat[len(results):]:
    ax.axis("off")
fig.suptitle("Dự đoán (màu) — nhãn COCO (trắng, nét chấm) — vùng cắt lượt 2 (vàng)", fontsize=11)
plt.tight_layout()
fig.savefig(OUT_DIR / "grid.png", dpi=120)
plt.show()

## So sánh với nhãn COCO (OKS)

OKS (Object Keypoint Similarity) là thước đo dùng trong COCO AP: 1 = trùng khớp hoàn toàn.
Chỉ tính cho ảnh lấy từ COCO.

In [ ]:
def oks(pred, gt, area):
    vis = gt[:, 2] > 0
    if not vis.any():
        return float("nan")
    d2 = ((pred[:, :2] - gt[:, :2]) ** 2).sum(1)
    e = d2 / (2 * (area + np.spacing(1)) * (2 * OKS_SIGMAS) ** 2)
    return float(np.exp(-e)[vis].mean())


rows = [(r["path"].name, oks(r["kp"], r["gt"]["kp"], r["gt"]["area"])) for r in results if r["gt"]]
if rows:
    for name, s in rows:
        print(f"{name:28s} OKS = {s:.3f}")
    print(f"{'Trung bình':28s} OKS = {np.nanmean([s for _, s in rows]):.3f}")
else:
    print("Không có ảnh COCO -> bỏ qua")

## Chi tiết một ảnh: độ tin cậy từng khớp

In [ ]:
IDX = 0  # chọn ảnh trong results
r = results[IDX]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw={"width_ratios": [1, 1.1]})
draw_pose(ax0, r["img"], r["kp"], r["box"], gt=r["gt"]["kp"] if r["gt"] else None, title=r["path"].name)
conf = r["kp"][:, 2]
ax1.barh(KEYPOINTS, conf, color=["tab:green" if c > CONF_THR else "tab:red" for c in conf])
ax1.axvline(CONF_THR, color="gray", ls="--", lw=1)
ax1.invert_yaxis(); ax1.set_xlim(0, 1); ax1.set_xlabel("độ tin cậy (đỉnh heatmap)")
plt.tight_layout()
plt.show()

for name, (x, y, c) in zip(KEYPOINTS, r["kp"]):
    print(f"{name:15s} x={x:7.1f}  y={y:7.1f}  conf={c:.3f}")

## Heatmap đầu ra của model

Ảnh đầu vào 256×256 (sau khi cắt vùng) và 17 heatmap 64×64 của từng khớp (đã gộp flip-test).

In [ ]:
@torch.no_grad()
def crop_heatmaps(img, box):
    M = square_affine(np.asarray(box[:2], np.float32), box[2], INPUT_SIZE)
    inp = cv2.warpAffine(img, M, (INPUT_SIZE, INPUT_SIZE), flags=cv2.INTER_LINEAR)
    x = torch.from_numpy(to_tensor_input(inp))[None].to(engine.device)
    if FLIP_TEST:
        x = torch.cat([x, torch.flip(x, [3])])
    x = x.half() if engine.half else x
    hm = engine.model(x.contiguous(memory_format=torch.channels_last)).float()
    if FLIP_TEST:
        hf = torch.flip(hm[1:], [3])[:, FLIP_INDEX]
        hf[:, :, :, 1:] = hf[:, :, :, :-1].clone()
        hm = (hm[:1] + hf) / 2
    return inp, hm[0].cpu().numpy()


inp, hm = crop_heatmaps(r["img"], r["box"])
fig, axes = plt.subplots(3, 6, figsize=(15, 7.5))
axes[0, 0].imshow(inp[:, :, ::-1]); axes[0, 0].set_title("input 256", fontsize=9)
axes[0, 0].imshow(cv2.resize(hm.max(0), (INPUT_SIZE, INPUT_SIZE)), alpha=0.5, cmap="jet")
for j, ax in enumerate(axes.flat[1:]):
    ax.imshow(inp[:, :, ::-1])
    ax.imshow(cv2.resize(hm[j], (INPUT_SIZE, INPUT_SIZE)), alpha=0.55, cmap="jet", vmin=0, vmax=max(hm.max(), 1e-6))
    ax.set_title(f"{KEYPOINTS[j]} ({hm[j].max():.2f})", fontsize=8)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Tốc độ suy luận

Đo thời gian 1 lần `predict_crop` (1 lượt, đã gồm flip-test nếu bật) trên ảnh đầu tiên.

In [ ]:
img = results[0]["img"]
box = full_box(img)
for _ in range(5):
    engine.predict_crop(img, box[:2], box[2])
N = 50
if engine.device == "cuda":
    torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(N):
    engine.predict_crop(img, box[:2], box[2])
if engine.device == "cuda":
    torch.cuda.synchronize()
ms = (time.perf_counter() - t0) / N * 1000
print(f"{engine.device} | flip={FLIP_TEST} | {ms:.2f} ms/lượt  (~{1000 / ms:.0f} FPS, 2 lượt/ảnh ~{500 / ms:.0f} FPS)")

## Lưu ảnh kết quả

In [ ]:
def render_cv2(img, kp, conf_thr=CONF_THR):
    out = img.copy()
    t = max(2, int(round(max(img.shape[:2]) / 300)))
    for (a, b), c in zip(SKELETON, LIMB_COLORS):
        if kp[a, 2] > conf_thr and kp[b, 2] > conf_thr:
            col = tuple(int(255 * v) for v in c[2::-1])  # RGBA -> BGR
            cv2.line(out, tuple(map(int, kp[a, :2])), tuple(map(int, kp[b, :2])), col, t, cv2.LINE_AA)
    for x, y, c in kp:
        if c > conf_thr:
            cv2.circle(out, (int(x), int(y)), t + 2, (255, 255, 255), -1, cv2.LINE_AA)
            cv2.circle(out, (int(x), int(y)), t + 2, (0, 0, 0), 1, cv2.LINE_AA)
    return out


for r in results:
    dst = OUT_DIR / f"{r['path'].stem}_pose.jpg"
    cv2.imwrite(str(dst), render_cv2(r["img"], r["kp"]))
print(f"Đã lưu {len(results)} ảnh + grid.png vào {OUT_DIR}")

## Thử nhanh một ảnh bất kỳ

Đổi `PATH` thành đường dẫn ảnh rồi chạy cell.

In [ ]:
PATH = images[0][0]  # ví dụ: ROOT / "test_images" / "me.jpg"

img = load_bgr(PATH)
kp, box, ms = detect(img)
fig, ax = plt.subplots(figsize=(7, 7))
draw_pose(ax, img, kp, box, title=f"{Path(PATH).name} — {ms:.0f} ms")
plt.show()